## 🎯 Learning Objectives
* Understand the core concepts of self-consistency and majority voting in LLM prompt engineering.
* Learn how to implement self-consistency with majority voting to improve LLM accuracy and robustness for complex reasoning tasks.
* Analyze the trade-offs between increased accuracy and computational cost when applying these techniques.
* Identify appropriate use cases for self-consistency and majority voting in real-world LLM applications.


## Self-consistency and Majority Voting: Enhancing LLM Reliability

Large Language Models (LLMs) are incredibly powerful, but they can sometimes struggle with complex reasoning tasks, leading to inconsistent or incorrect answers. This variability often stems from the probabilistic nature of their token generation. To combat this, advanced prompt engineering techniques like **Self-consistency** combined with **Majority Voting** have emerged as powerful tools to boost reliability and accuracy.

### What is Self-consistency?

Imagine you're asking a panel of experts to solve a challenging problem. Instead of just asking one expert once, you ask each expert independently to provide their reasoning and final answer. Self-consistency applies this same principle to LLMs. Instead of querying the LLM once, you prompt it multiple times for the same question, encouraging it to explore diverse reasoning paths. Each generation, even for the identical prompt, might yield a slightly different chain of thought or even a different final answer due to the model's inherent stochasticity (especially with a non-zero `temperature` setting).

### Why is it effective?

Complex problems often have multiple valid ways to arrive at a solution. By generating several reasoning paths, the LLM increases its chances of stumbling upon the correct one. It's like rolling a dice multiple times; the more rolls you have, the more likely you are to see a specific outcome, or in this case, a consistent and correct answer.

### How does Majority Voting come in?

Once you have multiple answers (and their associated reasoning paths) from the self-consistency process, you need a way to consolidate them into a single, most reliable answer. This is where **Majority Voting** shines. You simply count the occurrences of each unique final answer among the generated responses and select the one that appears most frequently. If there's a tie, you might choose one arbitrarily, or use a secondary heuristic (e.g., the answer with the shortest reasoning path, or the one that appears first).

### Step-by-Step Process:

1.  **Formulate the Prompt**: Create a clear and concise prompt for the complex reasoning task. This often includes instructions for step-by-step thinking (e.g., Chain-of-Thought prompting). For self-consistency, the prompt itself usually remains the same across multiple runs.
2.  **Generate Multiple Responses**: Query the LLM `N` times with the same prompt. Crucially, set the `temperature` parameter to a value greater than 0 (e.g., 0.7-1.0) to encourage diverse outputs and reasoning paths. Each response should ideally contain both the reasoning and the final answer.
3.  **Extract Final Answers**: From each of the `N` responses, parse and extract only the final answer. This might involve regular expressions or specific output formatting instructions in your prompt (e.g., "The final answer is: [ANSWER]").
4.  **Aggregate and Vote**: Collect all extracted final answers. Use a voting mechanism (like counting frequencies) to determine the most common answer. This majority-voted answer is then considered the self-consistent output.

This technique is particularly powerful for tasks requiring logical deduction, mathematical problem-solving, code generation, and other areas where a single 


In [ ]:
import random
from collections import Counter
import re
import time

# --- Mock LLM Class for Demonstration ---
# In a real scenario, this would be an API call to Google Gemini, OpenAI GPT, Anthropic Claude, etc.
class MockLLM:
    def __init__(self, model_name="Mock-LLM-2026", latency_ms=100):
        self.model_name = model_name
        self.latency_ms = latency_ms

    def generate(self, prompt: str, temperature: float = 0.7, max_tokens: int = 150) -> str:
        """
        Simulates an LLM response. For self-consistency, we need varied outputs.
        This mock LLM will sometimes make mistakes or give different reasoning.
        """
        time.sleep(self.latency_ms / 1000) # Simulate API latency

        # Define a set of possible answers and their likelihoods for a specific problem
        # Problem: "If a train travels at 60 mph for 2 hours, and then 40 mph for 1 hour, what is the total distance traveled?"
        possible_responses = [
            "First, 60 mph * 2 hours = 120 miles. Then, 40 mph * 1 hour = 40 miles. Total distance = 120 + 40 = 160 miles. The final answer is: 160 miles.",
            "The train travels 120 miles in the first part (60*2) and 40 miles in the second (40*1). So, 120 + 40 = 160 miles. The final answer is: 160 miles.",
            "Distance 1 = 60 * 2 = 120. Distance 2 = 40 * 1 = 40. Total = 120 + 40 = 160. The final answer is: 160 miles.",
            "It's 60*2 + 40*1 = 120 + 40 = 160. The final answer is: 160 miles.",
            "The first leg is 120 miles. The second leg is 40 miles. Total is 120 + 40 = 160. The final answer is: 160 miles.",
            "60 * 2 = 120. 40 * 1 = 40. Summing them up gives 160. The final answer is: 160 miles.",
            "The train travels 100 miles in total. (Incorrect reasoning for demonstration) The final answer is: 100 miles.",
            "120 + 40 = 160. The final answer is: 160 miles.",
            "The total distance is 180 miles. (Another incorrect reasoning) The final answer is: 180 miles."
        ]

        # Simulate variability based on temperature
        if temperature > 0.5:
            # Higher temperature, more likely to pick a diverse (potentially incorrect) answer
            return random.choice(possible_responses)
        else:
            # Lower temperature, more likely to pick a correct answer (simulated)
            return random.choice(possible_responses[:6]) # Pick from the mostly correct ones

# --- Self-consistency Implementation ---

def run_self_consistency(llm_client: MockLLM, prompt: str, num_generations: int = 5) -> str:
    """
    Applies self-consistency with majority voting to get a robust answer.
    """
    print(f"Running self-consistency with {num_generations} generations...")
    responses = []
    final_answers = []

    for i in range(num_generations):
        print(f"  Generating response {i+1}/{num_generations}...")
        # Use a non-zero temperature to encourage diverse reasoning paths
        response = llm_client.generate(prompt, temperature=0.8)
        responses.append(response)

        # Extract the final answer using a regex pattern
        match = re.search(r"The final answer is: (.*?)(?:\.|\n|$)", response)
        if match:
            final_answer = match.group(1).strip()
            final_answers.append(final_answer)
            print(f"    Extracted answer: {final_answer}")
        else:
            print(f"    Could not extract final answer from response {i+1}.")
            final_answers.append("N/A") # Handle cases where answer extraction fails

    print("\n--- All Generated Responses ---")
    for i, resp in enumerate(responses):
        print(f"Response {i+1}:\n{resp}\n---")

    print("\n--- Final Answers for Voting ---")
    print(final_answers)

    if not final_answers or all(ans == "N/A" for ans in final_answers):
        return "No consistent answer could be determined."

    # Majority Voting
    # Filter out 'N/A' answers before voting
    valid_answers = [ans for ans in final_answers if ans != "N/A"]
    if not valid_answers:
        return "No valid answers for voting."

    answer_counts = Counter(valid_answers)
    most_common_answer, count = answer_counts.most_common(1)[0]

    print(f"\n--- Voting Results ---")
    for answer, freq in answer_counts.items():
        print(f"  '{answer}': {freq} votes")

    print(f"\nMajority voted answer: {most_common_answer} (appeared {count} times)")
    return most_common_answer

# --- Main Execution ---
if __name__ == "__main__":
    llm = MockLLM()

    complex_reasoning_prompt = (
        "You are an expert in physics and mathematics. Solve the following problem step-by-step and state the final answer clearly. "
        "If a train travels at 60 mph for 2 hours, and then 40 mph for 1 hour, what is the total distance traveled? "
        "The final answer should be in the format: 'The final answer is: [NUMBER] miles.'"
    )

    # Run self-consistency with 7 generations
    final_result = run_self_consistency(llm, complex_reasoning_prompt, num_generations=7)
    print(f"\nFinal Self-Consistent Result: {final_result}")

    print("\n--- Comparison with a single generation (for context) ---")
    single_response = llm.generate(complex_reasoning_prompt, temperature=0.7)
    print(f"Single Generation Response:\n{single_response}")
    match_single = re.search(r"The final answer is: (.*?)(?:\.|\n|$)", single_response)
    single_answer = match_single.group(1).strip() if match_single else "N/A"
    print(f"Single Generation Answer: {single_answer}")


### Interpreting the Code Output and Performance Trade-offs

The code above demonstrates the self-consistency process. You'll observe several key aspects in the output:

1.  **Multiple Generations**: The `run_self_consistency` function makes `num_generations` calls to the `MockLLM`. Each call, due to the simulated `temperature > 0` and the `random.choice` in our mock, produces a slightly different response, including varied reasoning paths and sometimes even incorrect final answers.
2.  **Answer Extraction**: For each response, the code attempts to extract a standardized final answer using a regular expression. This highlights the importance of clear output formatting instructions in your prompt to facilitate automated parsing.
3.  **Voting Results**: The `Counter` object from `collections` efficiently tallies the occurrences of each unique extracted answer. You'll see how many times each answer appeared.
4.  **Majority Voted Answer**: The most frequently occurring answer is declared the self-consistent result. This is the core benefit: even if some individual generations were incorrect, the collective intelligence (majority vote) aims to converge on the correct answer.
5.  **Comparison**: The final part of the script shows a single generation for comparison. Notice how a single generation might yield an incorrect answer, whereas the self-consistent approach, by leveraging multiple attempts, is more likely to arrive at the correct one.

### Performance Trade-offs

While self-consistency significantly enhances accuracy and robustness for complex tasks, it comes with notable performance implications:

*   **Increased Computational Cost**: The most obvious trade-off is that you are querying the LLM `N` times instead of once. This directly translates to `N` times the API cost (if using commercial APIs) and `N` times the computational resources (if running models locally). For `num_generations=7`, you're paying for 7 prompts and 7 completions.
*   **Higher Latency**: Each generation takes time. Running `N` generations sequentially means your total response time will be approximately `N` times longer than a single query. This can be a critical factor in real-time applications.
*   **Complexity in Implementation**: You need robust parsing logic to extract answers from diverse LLM outputs, and a mechanism to handle cases where answers cannot be extracted or are ambiguous.

### Typical Use Cases

Given these trade-offs, self-consistency with majority voting is best suited for scenarios where:

*   **High Accuracy is Paramount**: Applications where even small errors can have significant consequences, such as medical diagnosis support, financial modeling, or critical infrastructure management.
*   **Complex Reasoning Tasks**: Problems involving multi-step logic, mathematical calculations, scientific simulations, or intricate code generation where LLMs are prone to 


### Resources for Further Learning

*   **Self-consistency Improves Chain of Thought Reasoning**: The original research paper by Wang et al. (2022) that introduced the self-consistency technique. [Link to arXiv paper](https://arxiv.org/abs/2203.11171)
*   **Google AI Studio / Gemini API Documentation**: Explore advanced prompting techniques and parameters like `temperature` for Google's Gemini models. [Google AI Studio](https://aistudio.google.com/) | [Gemini API Docs](https://ai.google.dev/docs)
*   **OpenAI API Documentation**: Learn about various prompting strategies and model parameters for GPT models. [OpenAI API Reference](https://platform.openai.com/docs/api-reference)
*   **Hugging Face Transformers Library**: A comprehensive library for working with various LLMs, including fine-tuning and inference. Useful for understanding how `temperature` and other generation parameters affect output. [Hugging Face Transformers Docs](https://huggingface.co/docs/transformers/index)
*   **Anthropic Claude API Documentation**: Information on prompting best practices and model capabilities for Claude models. [Anthropic Docs](https://docs.anthropic.com/claude/reference/getting-started-with-the-api)
